<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the core libraries needed for data handling and running the backtest.

In [ ]:
!pip install pandas zipline-reloaded

Zipline can require additional system-level dependencies (e.g., bcolz, TA-Lib) depending on your environment. If the pip install fails, consult the zipline-reloaded documentation for conda-based installation instructions.

## Imports and setup

pandas provides data structures for working with time series. zipline is a backtesting engine that simulates trading on historical data, handling order execution, portfolio tracking, and scheduling. We use zipline.pipeline to build a stock screening and ranking system that runs automatically each day before trading begins.

In [ ]:
import warnings

In [ ]:
import pandas as pd

In [ ]:
from zipline import run_algorithm
from zipline.api import (
    attach_pipeline,
    calendars,
    date_rules,
    get_open_orders,
    order_target_percent,
    pipeline_output,
    schedule_function,
    time_rules,
)
from zipline.pipeline import CustomFactor, Pipeline
from zipline.pipeline.data import USEquityPricing

Suppress warnings to keep the notebook output clean during the backtest.

In [ ]:
warnings.filterwarnings("ignore")

This constant controls how many stocks we hold at any given time. Equal-weighting across 10 positions means each stock gets 10% of the portfolio, which is a simple way to diversify without overcomplicating position sizing.

In [ ]:
N_LONGS = 10

## Define the momentum ranking factor

This custom factor computes momentum as the ratio of the most recent closing price to the oldest closing price in the lookback window. A value greater than 1 means the stock went up over that period.

In [ ]:
class Momentum(CustomFactor):
    inputs = [USEquityPricing.close]

    def compute(self, today, assets, out, close):
        out[:] = close[-1] / close[0]

Zipline's CustomFactor class lets us define reusable calculations that run across every stock in the universe each day. By dividing the last close by the first, we get a simple return ratio. This is the same core idea behind how professional momentum strategies rank stocks, though production systems often add volatility adjustments or sector controls on top.

The pipeline function creates our stock screening logic. We compute momentum over two timeframes and require both to be positive before considering a stock, then select the top performers by 30-day momentum.

In [ ]:
def make_pipeline():
    twenty_day_momentum = Momentum(window_length=20)
    thirty_day_momentum = Momentum(window_length=30)

    positive_momentum = (
        (twenty_day_momentum > 1)
        & (thirty_day_momentum > 1)
    )

    return Pipeline(
        columns={
            'longs': thirty_day_momentum.top(N_LONGS),
        },
        screen=positive_momentum,
    )

Requiring positive momentum on both 20-day and 30-day windows acts as a confirmation filter. A stock must be trending up over the shorter and longer period to qualify. This reduces the chance of buying a stock that spiked briefly but is already fading. The pipeline's screen parameter removes all stocks that fail this test before we ever rank them, which is how we turn the behavioral insight from the post (winners keep winning) into a concrete, testable rule.

## Set up the trading logic

This function runs every morning before the market opens. It pulls the latest pipeline output so our rebalance function always works with fresh rankings.

In [ ]:
def before_trading_start(context, data):
    context.factor_data = pipeline_output("factor_pipeline")
    assets = context.factor_data.index

The initialize function is called once at the start of the backtest. It attaches our pipeline and schedules the rebalance to run at market open every Monday.

In [ ]:
def initialize(context):
    attach_pipeline(make_pipeline(), "factor_pipeline")
    schedule_function(
        rebalance,
        date_rules.week_start(),
        time_rules.market_open(),
        calendar=calendars.US_EQUITIES,
    )

Weekly rebalancing is a deliberate choice. Rebalancing too often generates excessive trading costs, while rebalancing too rarely lets winning positions drift and losers linger. Weekly strikes a practical balance that most momentum research supports.

The rebalance function identifies which stocks to buy and which to sell. It sells any current holdings that no longer appear in the top momentum rankings, then allocates equal weight to each new winner.

In [ ]:
def rebalance(context, data):
    factor_data = context.factor_data
    assets = factor_data.index

    longs = assets[factor_data.longs]
    divest = context.portfolio.positions.keys() - longs

    exec_trades(data, assets=divest, target_percent=0)
    exec_trades(
        data,
        assets=longs,
        target_percent=1 / N_LONGS if N_LONGS else 0,
    )

The set subtraction on the divest line is worth understanding. It compares what we currently own against what the pipeline says we should own, and the difference is what we sell. This pattern ensures we only hold stocks that currently qualify, which prevents us from riding a position after its momentum has faded.

This helper function handles the actual order execution. It checks that each asset is tradeable and has no pending orders before placing a trade, which prevents duplicate orders and errors on delisted stocks.

In [ ]:
def exec_trades(data, assets, target_percent):
    for asset in assets:
        if data.can_trade(asset) and not get_open_orders(asset):
            order_target_percent(asset, target_percent)

Using order_target_percent instead of a fixed share count means zipline automatically calculates how many shares to buy or sell to reach the desired portfolio weight. This keeps our position sizes proportional to the portfolio value as it grows or shrinks over time.

## Run the momentum backtest

We define the backtest period and run the full simulation. Zipline processes every trading day from 2015 through 2018, executing our pipeline and rebalance logic as if we were trading live with $100,000.

In [ ]:
start = pd.Timestamp('2015')
end = pd.Timestamp('2018')

In [ ]:
perf = run_algorithm(
    start=start,
    end=end,
    initialize=initialize,
    before_trading_start=before_trading_start,
    capital_base=100_000,
    bundle="quandl",
)

The perf DataFrame returned by run_algorithm contains daily portfolio values, returns, and transaction details for the entire backtest period. This is where we can measure whether the momentum behavior we identified actually translated into returns. The "quandl" bundle provides historical US equity pricing data that zipline uses to simulate realistic fills and portfolio tracking.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.